<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 0. Settings
Signal QC of all 60 final sessions (from 01_sessions_setup: final_sessions.csv), both references (original, CMR).
Known bad channels (removed in preprocessing + extra_bad_channels.csv) are left out before the CMR.
Shared functions in main\functions.py (imported as fn). RUN = False in the loop: nothing is computed until it is switched on.
Per-session results -> 2_processed\<session>\amir_data\QC; summary tables -> main\tables; figures -> main\figures\04_QC.

<div>

In [1]:
# %%
# ── 0. SETTINGS ──
import os, sys, importlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MAIN      = r"C:\Users\ar687\Documents\GitHub\AmirAnalysis\main"
sys.path.insert(0, MAIN)
import functions as fn
importlib.reload(fn)                                   # picks up changes in functions.py without restarting

DATA_ROOT = r"O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data"
PROC      = os.path.join(DATA_ROOT, "2_processed")
AMIR      = os.path.join(DATA_ROOT, "amir_data")
TABLES    = os.path.join(MAIN, "tables")
FIGS      = os.path.join(MAIN, "figures", "04_QC")
SUB       = os.path.join("amir_data", "down_sampled", "1000_Hz")
QC_SUB    = os.path.join("amir_data", "QC")              # per-session QC files
os.makedirs(FIGS, exist_ok=True)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

final = pd.read_csv(os.path.join(TABLES, "final_sessions.csv"), dtype={"rat": str})
XB    = pd.read_csv(os.path.join(TABLES, "extra_bad_channels.csv"))          # read only here
REFS  = ["original", "CMR"]

print(f"final sessions: {len(final)} | extra bad channel rows: {len(XB)} | functions.py loaded from {fn.__file__}")

final sessions: 60 | extra bad channel rows: 51 | functions.py loaded from C:\Users\ar687\Documents\GitHub\AmirAnalysis\main\functions.py


<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 1. Load one session (test)
fn.load_session: 1000 Hz recording (channels removed in preprocessing already missing) minus the session's extra bad channels
(extra_bad_channels.csv), plus the CMR version (global median, on the fly). fn.probe_table: position and shank of each channel.
Test on the two pilots: 136_0529 (only AP138 removed in preprocessing, no extra) and 135_0529 (7 in preprocessing + AP362, AP381).

<div>

In [2]:
# %%
# ── 1. LOAD ONE SESSION (test on the two pilots) ──
importlib.reload(fn)
for s in ["20260529_r0000136_wt_1318", "20260529_r0000135_wt_1411"]:
    recs, removed = fn.load_session(PROC, s, XB)
    pdf = fn.probe_table(recs["original"])
    print(f"{s}: channels {recs['original'].get_num_channels()} (original) / {recs['CMR'].get_num_channels()} (CMR) "
          f"| extra bad removed: {removed} | fs {recs['original'].get_sampling_frequency():.0f} Hz "
          f"| {recs['original'].get_total_duration() / 60:.1f} min | shanks: {pdf['shank'].value_counts().sort_index().to_dict()}")

20260529_r0000136_wt_1318: channels 383 (original) / 383 (CMR) | extra bad removed: [] | fs 1000 Hz | 32.1 min | shanks: {0: 119, 1: 187, 2: 77}
20260529_r0000135_wt_1411: channels 375 (original) / 375 (CMR) | extra bad removed: ['imec0.ap#AP362', 'imec0.ap#AP381'] | fs 1000 Hz | 41.2 min | shanks: {0: 28, 1: 201, 2: 146}


<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 2. PSD per channel (test on one session)
fn.compute_psd: Welch PSD per channel from 30 pieces of 10 s spread over the whole recording (2 s windows, 0.5 Hz, 1–100 Hz).
fn.psd_table: per channel log10 power per band (delta 1–3, theta 4–7, alpha 8–12, beta 13–29, low gamma 30–60, high gamma 61–100),
broadband power, and the 50 Hz line ratio (49–51 Hz vs 45–48 + 52–55 Hz; ~1 or below = notch worked).
Both references. Test session: 20260529_r0000136_wt_1318 (pilot; earlier result: 50 Hz fine).

<div>

In [3]:
# %%
# ── 2. PSD PER CHANNEL (test on one session; the loop over all sessions is step 4) ──
importlib.reload(fn)
TEST = "20260529_r0000136_wt_1318"
recs, removed = fn.load_session(PROC, TEST, XB)
pdf = fn.probe_table(recs["original"])
psd, qc = {}, {}
for ref in REFS:
    freqs, psd[ref], ids = fn.compute_psd(recs[ref])
    qc[ref] = fn.psd_table(freqs, psd[ref], ids, pdf)
    print(f"{ref:8s}: {len(ids)} channels | freqs {freqs[0]:.1f}–{freqs[-1]:.1f} Hz ({len(freqs)} bins) | "
          f"50 Hz ratio median {qc[ref]['line_ratio'].median():.2f}, max {qc[ref]['line_ratio'].max():.2f} | "
          f"median log broadband {qc[ref]['log_broadband'].median():.2f}")

original: 383 channels | freqs 1.0–100.0 Hz (199 bins) | 50 Hz ratio median 0.21, max 0.23 | median log broadband 1.73
CMR     : 383 channels | freqs 1.0–100.0 Hz (199 bins) | 50 Hz ratio median 0.23, max 0.26 | median log broadband 1.30


<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 3. Single bad channels + 50 Hz check (test on one session)
fn.flag_channels_local: each channel's broadband power vs the median of its neighbours on the same shank (±75 µm, >= 3 neighbours);
flagged if |Δlog10| > 0.3 (factor 2). Depth blocks are not flagged, only lone odd channels.
50 Hz: flagged if line_ratio > 1 (more power at 49–51 Hz than next to it = notch did not remove the line).
Earlier result for 20260529_r0000136_wt_1318 (2026-10-19): original 0 outliers; CMR 3 (AP17, AP324, AP326, ~2.5x more power), kept.

<div>

In [4]:
# %%
# ── 3. SINGLE BAD CHANNELS + 50 Hz (test session from step 2) ──
importlib.reload(fn)
for ref in REFS:
    loc = fn.flag_channels_local(qc[ref], radius_um=75, thr_log=0.3)
    line = qc[ref][qc[ref]["line_ratio"] > 1]
    print(f"\n{ref}: {len(loc)} local outliers | {len(line)} channels with 50 Hz line ratio > 1")
    if len(loc):
        print(loc[["channel_id", "shank", "y", "local_dev"]].round(2).to_string(index=False))
    if len(line):
        print(line[["channel_id", "shank", "y", "line_ratio"]].round(2).to_string(index=False))


original: 0 local outliers | 0 channels with 50 Hz line ratio > 1

CMR: 3 local outliers | 0 channels with 50 Hz line ratio > 1
    channel_id  shank      y  local_dev
 imec0.ap#AP17      0 3000.0       0.42
imec0.ap#AP324      1 2070.0       0.40
imec0.ap#AP326      1 2085.0       0.40


<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 4. Signal QC of all final sessions (loop)
Per session: fn.load_session (minus extra bad channels) -> fn.qc_session for original and CMR
(PSD 30 x 10 s, band powers, 50 Hz ratio, local_dev vs neighbours ±75 µm).
Saved per session: 2_processed\<session>\amir_data\QC\psd.npz (freqs, PSD per reference, channel ids) and qc_channels.csv.
All sessions together: tables\qc_channels_all.csv (one row per session x reference x channel).
RUN = False: only reports which sessions are done. Sessions with a finished QC file are skipped (OVERWRITE = True to redo).
Estimated time: ~1–2 min per session (not measured), ~1–2 h for 60 sessions.

<div>

In [8]:
# %%
# ── 4. SIGNAL QC OF ALL FINAL SESSIONS (loop) ──
importlib.reload(fn)
RUN, OVERWRITE = True, False
ERR_LOG = os.path.join(TABLES, "qc_errors.csv")

def qc_files(s):
    d = os.path.join(PROC, s, QC_SUB)
    return d, os.path.join(d, "psd.npz"), os.path.join(d, "qc_channels.csv")

done = [s for s in final["session"] if os.path.exists(qc_files(s)[2])]
todo = [s for s in final["session"] if OVERWRITE or s not in done]
print(f"final sessions: {len(final)} | QC done: {len(done)} | to do: {len(todo)}")

if RUN:
    t_all = time.time() if "time" in globals() else None
    import time
    t_all = time.time()
    for i, s in enumerate(todo, 1):
        t0 = time.time()
        try:
            recs, removed = fn.load_session(PROC, s, XB)
            pdf = fn.probe_table(recs["original"])
            freqs, psd, tab = fn.qc_session(recs, pdf)
            tab.insert(0, "session", s); tab.insert(1, "rat", final.set_index("session").loc[s, "rat"])
            d, f_npz, f_csv = qc_files(s)
            os.makedirs(d, exist_ok=True)
            np.savez_compressed(f_npz, freqs=freqs, channel_ids=np.array(pdf["channel_id"]),
                                **{f"psd_{r}": p for r, p in psd.items()})
            tab.to_csv(f_csv, index=False)                                   # written last = session finished
            n_out = tab.groupby("ref")["flag_local"].sum().to_dict()
            print(f"[{i}/{len(todo)}] {s}: {(time.time() - t0) / 60:.1f} min | local outliers {n_out} | "
                  f"max 50 Hz ratio {tab['line_ratio'].max():.2f}", flush=True)
        except Exception as e:                                               # one broken session does not stop the run
            print(f"[{i}/{len(todo)}] {s}: ERROR {e!r}", flush=True)
            pd.DataFrame([{"session": s, "error": repr(e)}]).to_csv(
                ERR_LOG, mode="a", header=not os.path.exists(ERR_LOG), index=False)
    print(f"\nfinished in {(time.time() - t_all) / 60:.0f} min")

# collect all finished sessions into one table (also when RUN = False)
parts = [pd.read_csv(qc_files(s)[2], dtype={"rat": str}) for s in final["session"] if os.path.exists(qc_files(s)[2])]
if parts:
    QC = pd.concat(parts, ignore_index=True)
    QC.to_csv(os.path.join(TABLES, "qc_channels_all.csv"), index=False)
    print(f"qc_channels_all.csv: {QC['session'].nunique()} sessions, {len(QC)} rows")
else:
    print("no QC results yet (RUN = False)")

final sessions: 60 | QC done: 0 | to do: 60
[1/60] 20260510_r0000135_wt_1625: 0.2 min | local outliers {'CMR': 9, 'original': 0} | max 50 Hz ratio 0.82
[2/60] 20260510_r0000136_wt_1926: 0.2 min | local outliers {'CMR': 1, 'original': 0} | max 50 Hz ratio 0.31
[3/60] 20260511_r0000136_wt_0909: 0.2 min | local outliers {'CMR': 5, 'original': 0} | max 50 Hz ratio 0.31
[4/60] 20260511_r0000136_wt_0930: 0.3 min | local outliers {'CMR': 3, 'original': 0} | max 50 Hz ratio 0.30
[5/60] 20260511_r0000136_wt_1832: 0.2 min | local outliers {'CMR': 1, 'original': 0} | max 50 Hz ratio 0.33
[6/60] 20260512_r0000136_wt_1054: 0.2 min | local outliers {'CMR': 1, 'original': 0} | max 50 Hz ratio 0.35
[7/60] 20260513_r0000136_wt_1729: 0.2 min | local outliers {'CMR': 2, 'original': 0} | max 50 Hz ratio 0.31
[8/60] 20260514_r0000136_wt_1317: 0.2 min | local outliers {'CMR': 6, 'original': 1} | max 50 Hz ratio 0.30
[9/60] 20260515_r0000135_wt_1953: 0.2 min | local outliers {'CMR': 31, 'original': 0} | max 

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 5. QC summary (all final sessions)
From tables\qc_channels_all.csv (step 4). Per session and reference: channels, local outliers (|Δlog10| > 0.3 vs neighbours),
channels with 50 Hz ratio > 1, median 50 Hz ratio. Per rat: sums. Repeat offenders: per rat and reference, readout channels (AP number)
and electrode positions (x, y) flagged in >= 3 sessions and >= 30 % of the sessions that recorded them.
Saved: tables\qc_sessions.csv, tables\qc_repeat_outliers.csv.

<div>

In [9]:
# %%
# ── 5. QC SUMMARY: per session, per rat, repeat offenders ──
f_all = os.path.join(TABLES, "qc_channels_all.csv")
if not os.path.exists(f_all):
    print("no QC results yet - run step 4 with RUN = True first")
else:
    QC = pd.read_csv(f_all, dtype={"rat": str})
    QC["ap"] = QC["channel_id"].astype(str).str.extract(r"(AP\d+)")[0]
    QC["line_flag"] = QC["line_ratio"] > 1

    # per session x reference
    ses = (QC.groupby(["rat", "session", "ref"])
             .agg(channels=("channel_id", "size"), local_outliers=("flag_local", "sum"),
                  line_gt1=("line_flag", "sum"), line_ratio_median=("line_ratio", "median"))
             .reset_index())
    ses.to_csv(os.path.join(TABLES, "qc_sessions.csv"), index=False)
    print(f"sessions with QC: {ses['session'].nunique()} of {len(final)}")
    print("\nper rat and reference (sum over sessions; median line ratio = median over sessions):")
    print(ses.groupby(["rat", "ref"]).agg(sessions=("session", "nunique"), local_outliers=("local_outliers", "sum"),
                                          max_per_session=("local_outliers", "max"), line_gt1=("line_gt1", "sum"),
                                          line_ratio_median=("line_ratio_median", "median")).round(2).to_string())
    print("\nsessions with the most local outliers:")
    print(ses.sort_values("local_outliers", ascending=False).head(10).to_string(index=False))

    # repeat offenders: same readout channel / same electrode position flagged in many sessions of a rat
    rows = []
    for key, name in [(["ap"], "readout channel"), (["x", "y"], "electrode position")]:
        g = QC.groupby(["rat", "ref"] + key).agg(n_sessions=("session", "nunique"), n_flagged=("flag_local", "sum"),
                                                 median_dev=("local_dev", "median")).reset_index()
        g["pct_flagged"] = (100 * g["n_flagged"] / g["n_sessions"]).round(0)
        g = g[(g["n_flagged"] >= 3) & (g["pct_flagged"] >= 30)]
        g.insert(0, "by", name)
        rows.append(g)
    rep = pd.concat(rows, ignore_index=True).sort_values(["by", "rat", "ref", "n_flagged"], ascending=[True, True, True, False])
    rep.to_csv(os.path.join(TABLES, "qc_repeat_outliers.csv"), index=False)
    print(f"\nrepeat offenders (flagged in >= 3 sessions and >= 30 % of the sessions recording them): {len(rep)}")
    print(rep.round(2).to_string(index=False) if len(rep) else "none")

sessions with QC: 60 of 60

per rat and reference (sum over sessions; median line ratio = median over sessions):
              sessions  local_outliers  max_per_session  line_gt1  line_ratio_median
rat ref                                                                             
134 CMR             11              41                9         0               0.24
    original        11               0                0         0               0.21
135 CMR             29             219               34         0               0.23
    original        29               0                0         8               0.37
136 CMR             20              49                9         0               0.23
    original        20               2                1         0               0.21

sessions with the most local outliers:
rat                   session ref  channels  local_outliers  line_gt1  line_ratio_median
135 20260515_r0000135_wt_1957 CMR       376              34         0         

<div style="color:#2e7d32; border-left:4px solid #2e7d32; padding-left:10px">

## 6. QC plots for one session (on request)
Set PLOT_SESSION. From the saved spectra (step 4): PSD by depth per shank (top: log10 PSD; bottom: minus shank median,
so single odd channels stand out) for original and CMR, and the median PSD original vs CMR.
From the recording: 4 s of raw LFP from 20 channels (fn.select_channels), middle of the session.
Saved: figures\04_QC\<session>\.

<div>

In [11]:
# %%
# ── 6. QC PLOTS FOR ONE SESSION -> saved in 2_processed\<session>\amir_data\QC, not shown ──
importlib.reload(fn)
PLOT_SESSION = "20260529_r0000136_wt_1318"

d, f_npz, f_csv = qc_files(PLOT_SESSION)
if not os.path.exists(f_npz):
    print(f"no QC file for {PLOT_SESSION} - run step 4 first")
else:
    z = np.load(f_npz, allow_pickle=True)
    freqs, ids = z["freqs"], [str(c) for c in z["channel_ids"]]
    psd = {r: z[f"psd_{r}"] for r in REFS}
    recs, _ = fn.load_session(PROC, PLOT_SESSION, XB)
    pdf = fn.probe_table(recs["original"])
    sel = fn.select_channels(pdf, n_total=20)
    for ref in REFS:
        fn.plot_psd_depth(freqs, psd[ref], ids, pdf, f"{PLOT_SESSION} — PSD by depth ({ref})",
                          save_path=os.path.join(d, f"psd_depth_{ref.lower()}.png"), show=False)
    fn.plot_mean_psd(psd, freqs, f"{PLOT_SESSION} — median PSD over channels",
                     save_path=os.path.join(d, "psd_original_vs_cmr.png"), show=False)
    t0 = recs["original"].get_total_duration() / 2
    for ref in REFS:
        fn.plot_traces(recs[ref], sel, t0, 4, f"{PLOT_SESSION} — raw LFP ({ref})",
                       save_path=os.path.join(d, f"traces_{ref.lower()}.png"), show=False)
    print("saved in", d, ":", sorted(f for f in os.listdir(d) if f.endswith(".png")))

Saved: O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data\2_processed\20260529_r0000136_wt_1318\amir_data\QC\psd_depth_original.png
Saved: O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data\2_processed\20260529_r0000136_wt_1318\amir_data\QC\psd_depth_cmr.png
Saved: O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data\2_processed\20260529_r0000136_wt_1318\amir_data\QC\psd_original_vs_cmr.png
Saved: O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data\2_processed\20260529_r0000136_wt_1318\amir_data\QC\traces_original.png
Saved: O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data\2_processed\20260529_r0000136_wt_1318\amir_data\QC\traces_cmr.png
saved in O:\archive\projects\2020_euSNN\Neuropixels_RT_mPFC\data\2_processed\20260529_r0000136_wt_1318\amir_data\QC : ['psd_depth_cmr.png', 'psd_depth_original.png', 'psd_original_vs_cmr.png', 'traces_cmr.png', 'traces_original.png']
